### Face detection model

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import IPython.display
import degirum as dg, degirum_tools, mytools
lap = degirum_tools.import_optional_package("lap")
cython_bbox = degirum_tools.import_optional_package("cython_bbox")

In [ ]:
target = dg.CLOUD 
zoo_ = dg.connect(target, "https://cs.degirum.com/degirum/openvino", degirum_tools.get_token())
zoo = dg.connect(target, "https://cs.degirum.com/degirum/yolov8_sandbox", degirum_tools.get_token())

In [ ]:
input_filename = './videoplayback.mp4'
model_name = "yolov8n_relu6_face--640x640_quant_openvino_cpu_2"
model_name_age = "age_gender_recognition_retail--62x62_float_openvino_cpu_1"
# model_name = "yolov8n_relu6_widerface--512x512_quant_n2x_orca1_1"
model = zoo.load_model(model_name)
model_age = zoo_.load_model(model_name_age)

# set model parameters
model.overlay_show_probabilities = True
model.overlay_line_width = 1
model.image_backend = "opencv"

In [ ]:
orig_path = Path(input_filename)
ann_path = "./face_detected" + orig_path.stem + "_annotated_age_gender" + orig_path.suffix

In [ ]:
orig_path.stem,orig_path.suffix,ann_path

In [ ]:
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 0.4
font_thickness = 1

with degirum_tools.open_video_stream(input_filename) as stream:
    
    image_w = int(stream.get(cv2.CAP_PROP_FRAME_WIDTH))
    image_h = int(stream.get(cv2.CAP_PROP_FRAME_HEIGHT))
    degirum_tools.video_source(stream)
    print (image_w,image_h)
    
    with degirum_tools.Display("MoT") as display, \
         degirum_tools.open_video_writer(str(ann_path), image_w, image_h) as writer:
        fps = 30
        count = 0
        progress = degirum_tools.Progress(int(stream.get(cv2.CAP_PROP_FRAME_COUNT)))
        for batch_result in model.predict_batch(degirum_tools.video_source(stream)):
            results = batch_result.results

            bboxes = np.zeros((len(results), 5))
            image = batch_result.image
            for index, result in enumerate(results):
                bbox = np.array(result.get('bbox', [0, 0, 0, 0]))
                boxes = bbox.astype(int)
                score = result.get('score', 0)
                bbox_and_score = np.append(bbox, score)
                bboxes[index] = bbox_and_score
                box1 = tuple(boxes[0:2])
                box2 = tuple(boxes[2:4])
                facebox = bbox
                facewidth, faceheight = (facebox[2] - facebox[0]), (facebox[3] - facebox[1])
                crop = image[int(facebox[1]):int(facebox[1]+faceheight), int(facebox[0]):int(facebox[0]+facewidth)]
                cv2.rectangle(image, box1, box2, color=(0, 255, 0), thickness=1)
                res_age = model_age(crop)
                predictions= res_age.results[0]["results"]
                gender_pred = max(predictions, key=lambda x: x['score'])
                age = res_age.results[1]["results"][0]["label"]
                gender_and_age = gender_pred['label']+":" + str(age)
                
                text_size = cv2.getTextSize(gender_and_age, font, font_scale, font_thickness)[0]
                text_position = (int((box1[0] + box2[0] - text_size[0]) / 2), box1[1] - 5)
                cv2.putText(image, gender_and_age, text_position, font, font_scale, (0, 0, 255), font_thickness, cv2.LINE_AA)
                
    
            writer.write(image)
#             display.show(image)
            progress.step()
